# clustering_squidpy parameter tuning

This notebook calls the same functions used by the Nextflow `CLUSTERING_SQUIDPY` stage so parameters can be adjusted interactively before committing them to the pipeline config.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from itertools import combinations, product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.metrics import adjusted_rand_score

from IPython.display import Image, display
from tqdm.auto import tqdm

from merxen.analysis.clustering_squidpy import (
    load_spatialdata_adata,
    plot_qc_histograms,
    plot_spatial_scatter,
    plot_umap,
    remove_control_features,
    run_clustering_squidpy,
    run_scanpy_clustering,
    save_clustered_adata,
    save_qc_metrics,
)
from merxen.config import (
    ClusteringSquidpyConfig,
    ClusteringSquidpySampleConfig,
)


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

pair_id = "P7513"
results_dir = PROJECT_ROOT / "results"
output_dir = results_dir / pair_id / "clustering_squidpy_interactive"

samples = [
    ClusteringSquidpySampleConfig(
        sample_id=f"{pair_id}_MERSCOPE",
        platform="MERSCOPE",
        zarr_path=results_dir
        / pair_id
        / "merscope"
        / "latest"
        / "latest_spatialdata.zarr",
    ),
    ClusteringSquidpySampleConfig(
        sample_id=f"{pair_id}_XENIUM",
        platform="XENIUM",
        zarr_path=results_dir
        / pair_id
        / "xenium"
        / "latest"
        / "latest_spatialdata.zarr",
    ),
]

cfg = ClusteringSquidpyConfig(
    pair_id=pair_id,
    output_dir=output_dir,
    samples=samples,
    drop_control_features=True,
    min_counts=10,
    min_cells=5,
    normalize_target_sum=None,
    normalize_exclude_highly_expressed=False,
    normalize_max_fraction=0.05,
    n_pcs=60,
    n_neighbors=30,
    leiden_resolution=0.5,
    umap_min_dist=0.3,
    umap_spread=1.0,
    random_seed=0,
    spatial_point_size=0.5,
    figure_dpi=180,
    use_gpu=True,
)
cfg


Load one sample and build the Squidpy-ready AnnData object.

In [ ]:
sample = cfg.samples[0]
sample_out = cfg.output_dir / sample.platform.lower()
sample_out.mkdir(parents=True, exist_ok=True)

adata = load_spatialdata_adata(
    sample.zarr_path,
    platform=sample.platform,
    table_key=sample.table_key,
    shape_key=sample.shape_key,
)
adata

Save the same QC outputs as the pipeline.

In [ ]:
qc_plot = plot_qc_histograms(
    adata,
    sample_out / f"{sample.sample_id}_qc_histograms.png",
    sample_label=sample.sample_id,
    platform=sample.platform,
    dpi=cfg.figure_dpi,
)
save_qc_metrics(adata, sample_out / f"{sample.sample_id}_qc_metrics.csv")
display(Image(filename=qc_plot))

In [ ]:
adata

Run the exact Scanpy clustering function with the pipeline arguments. Change `cfg.min_counts`, `cfg.min_cells`, `cfg.n_neighbors`, or `cfg.leiden_resolution` above and rerun from here.

In [ ]:
clustered = run_scanpy_clustering(
    adata,
    drop_control_features=cfg.drop_control_features,
    min_counts=cfg.min_counts,
    min_cells=cfg.min_cells,
    normalize_target_sum=cfg.normalize_target_sum,
    normalize_exclude_highly_expressed=cfg.normalize_exclude_highly_expressed,
    normalize_max_fraction=cfg.normalize_max_fraction,
    n_pcs=cfg.n_pcs,
    n_neighbors=cfg.n_neighbors,
    leiden_resolution=cfg.leiden_resolution,
    umap_min_dist=cfg.umap_min_dist,
    umap_spread=cfg.umap_spread,
    random_seed=cfg.random_seed,
    use_gpu=cfg.use_gpu,
)
clustered


In [ ]:
adata

Save the UMAP, spatial scatter, and clustered AnnData exactly as the pipeline does.

In [ ]:
umap_plot = plot_umap(
    clustered,
    sample_out / f"{sample.sample_id}_umap.png",
    dpi=cfg.figure_dpi,
)
spatial_plot = plot_spatial_scatter(
    clustered,
    sample_out / f"{sample.sample_id}_spatial_scatter_leiden.png",
    point_size=1,#cfg.spatial_point_size,
    dpi=cfg.figure_dpi,
)
save_clustered_adata(
    clustered,
    sample_out / f"{sample.sample_id}_clustered.h5ad",
)
display(Image(filename=umap_plot))
display(Image(filename=spatial_plot))

## Interactive Parameter Tuning

Run the cells below one at a time. They reuse `adata` loaded above and call the same `run_scanpy_clustering()` function as the pipeline, changing only one family of parameters at a time. The two panels in each sweep are UMAP and physical-space scatter colored by Leiden cluster.


### Practical Tuning Notes

- `n_pcs`: choose enough PCs to cover the elbow/cumulative variance plateau, then confirm clusters are not driven by trailing noise. With a 300-gene panel, values like 20, 30, 50, and 80 are reasonable probes.
- `n_neighbors`: smaller values preserve local structure and can split fine states; larger values smooth toward global structure. Scanpy's default is 15 and docs suggest roughly 2-100 as the useful range.
- `leiden_resolution`: higher means more clusters. Sweep it after choosing `n_pcs` and `n_neighbors`, then prefer resolutions where clusters are interpretable, spatially coherent, and not dominated by tiny fragments.
- `umap_min_dist` and `umap_spread`: these affect the visualization layout, not the neighbor graph or Leiden labels. Low `min_dist` makes tighter islands; high `min_dist` gives a more diffuse view.
- `normalize_target_sum`: after `log1p`, it mostly changes scale/offset when all cells are normalized consistently; it matters more if you compare absolute normalized values or mix datasets.
- `normalize_exclude_highly_expressed`: leave `False` by default for a targeted Xenium/MERSCOPE panel unless one/few genes dominate counts and distort size factors.


In [ ]:
tuning_dir = sample_out / "parameter_tuning"
tuning_dir.mkdir(parents=True, exist_ok=True)

base_params = dict(
    drop_control_features=cfg.drop_control_features,
    min_counts=cfg.min_counts,
    min_cells=cfg.min_cells,
    normalize_target_sum=cfg.normalize_target_sum,
    normalize_exclude_highly_expressed=cfg.normalize_exclude_highly_expressed,
    normalize_max_fraction=cfg.normalize_max_fraction,
    n_pcs=cfg.n_pcs,
    n_neighbors=cfg.n_neighbors,
    leiden_resolution=cfg.leiden_resolution,
    umap_min_dist=cfg.umap_min_dist,
    umap_spread=cfg.umap_spread,
    random_seed=cfg.random_seed,
    use_gpu=cfg.use_gpu,
)


def cluster_with(**overrides):
    params = base_params | overrides
    return run_scanpy_clustering(adata, **params)


def cluster_summary(adata_clustered, label):
    counts = adata_clustered.obs["leiden"].value_counts().sort_index()
    params = dict(adata_clustered.uns.get("merxen_clustering_params", {}))
    return {
        "label": label,
        "n_cells": int(adata_clustered.n_obs),
        "n_genes": int(adata_clustered.n_vars),
        "n_clusters": int(counts.size),
        "min_cluster_size": int(counts.min()) if counts.size else 0,
        "median_cluster_size": float(counts.median()) if counts.size else np.nan,
        "max_cluster_size": int(counts.max()) if counts.size else 0,
        **params,
    }


def plot_umap_and_spatial_grid(results, *, title, save_name=None):
    fig, axes = plt.subplots(
        len(results),
        2,
        figsize=(12, max(3.6, 3.5 * len(results))),
        squeeze=False,
    )
    summary = []
    for row, (label, clustered_i) in enumerate(tqdm(results, desc=title)):
        summary.append(cluster_summary(clustered_i, label))
        sc.pl.umap(
            clustered_i,
            color="leiden",
            ax=axes[row, 0],
            title=f"UMAP: {label}",
            legend_loc="on data",
            show=False,
        )
        coords = clustered_i.obsm["spatial"]
        labels = clustered_i.obs["leiden"].astype("category")
        axes[row, 1].scatter(
            coords[:, 0],
            coords[:, 1],
            c=labels.cat.codes,
            s=cfg.spatial_point_size,
            cmap="tab20",
            edgecolors="none",
            linewidths=0,
            rasterized=True,
        )
        axes[row, 1].set_title(f"Spatial: {label}")
        axes[row, 1].set_aspect("equal")
        axes[row, 1].invert_yaxis()
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    if save_name:
        fig.savefig(tuning_dir / save_name, dpi=cfg.figure_dpi, bbox_inches="tight")
    display(pd.DataFrame(summary))
    display(fig)
    return fig, pd.DataFrame(summary)


def mean_pairwise_ari(clusterings):
    values = [
        adjusted_rand_score(a, b)
        for a, b in combinations(clusterings, 2)
    ]
    return float(np.mean(values)) if values else np.nan


### PCA Diagnostics

Use this first to decide the upper range for `n_pcs`. The elbow plot shows variance explained by each PC; the cumulative plot helps avoid choosing so few PCs that you discard meaningful structure.


In [ ]:
pca_probe = remove_control_features(adata) if cfg.drop_control_features else adata.copy()
sc.pp.filter_cells(pca_probe, min_counts=cfg.min_counts)
sc.pp.filter_genes(pca_probe, min_cells=cfg.min_cells)
pca_probe.layers["counts"] = pca_probe.X.copy()
sc.pp.normalize_total(
    pca_probe,
    target_sum=cfg.normalize_target_sum,
    exclude_highly_expressed=cfg.normalize_exclude_highly_expressed,
    max_fraction=cfg.normalize_max_fraction,
    inplace=True,
)
sc.pp.log1p(pca_probe)
max_probe_pcs = min(100, pca_probe.n_obs - 1, pca_probe.n_vars - 1)
sc.pp.pca(pca_probe, n_comps=max_probe_pcs, random_state=cfg.random_seed)

variance_ratio = pca_probe.uns["pca"]["variance_ratio"]
cumulative = np.cumsum(variance_ratio)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np.arange(1, len(variance_ratio) + 1), variance_ratio, marker="o", ms=3)
axes[0].set_xlabel("PC")
axes[0].set_ylabel("Variance ratio")
axes[0].set_title("PCA elbow")
axes[1].plot(np.arange(1, len(cumulative) + 1), cumulative, marker="o", ms=3)
axes[1].axhline(0.8, color="tab:orange", ls="--", lw=1)
axes[1].axhline(0.9, color="tab:red", ls="--", lw=1)
axes[1].set_xlabel("PC")
axes[1].set_ylabel("Cumulative variance ratio")
axes[1].set_title("Cumulative variance")
fig.tight_layout()
fig.savefig(tuning_dir / f"{sample.sample_id}_pca_diagnostics.png", dpi=cfg.figure_dpi, bbox_inches="tight")
display(fig)

pd.DataFrame(
    {
        "pc": np.arange(1, len(variance_ratio) + 1),
        "variance_ratio": variance_ratio,
        "cumulative_variance_ratio": cumulative,
    }
).head(25)


### Sweep `n_pcs`

Keep neighbors, resolution, and UMAP layout fixed. Look for cluster labels and spatial patterns that remain coherent as PCs increase.


In [ ]:
n_pcs_grid = [10, 20, 30, 50, 80]
n_pcs_results = []
for n_pcs in n_pcs_grid:
    max_allowed = min(adata.n_obs - 1, adata.n_vars - 1)
    if n_pcs <= 0 or n_pcs > max_allowed:
        continue
    n_pcs_results.append((f"n_pcs={n_pcs}", cluster_with(n_pcs=n_pcs)))

_, n_pcs_summary = plot_umap_and_spatial_grid(
    n_pcs_results,
    title="n_pcs sweep",
    save_name=f"{sample.sample_id}_n_pcs_sweep.png",
)


### Sweep `n_neighbors`

Small values emphasize local neighborhoods and can over-fragment; large values smooth toward broad structure. Start around 10-50 for this panel and expand if needed.


In [ ]:
n_neighbors_grid = [5, 10, 15, 30, 50]
neighbors_results = []
for n_neighbors in n_neighbors_grid:
    if n_neighbors >= adata.n_obs:
        continue
    neighbors_results.append((f"n_neighbors={n_neighbors}", cluster_with(n_neighbors=n_neighbors)))

_, neighbors_summary = plot_umap_and_spatial_grid(
    neighbors_results,
    title="n_neighbors sweep",
    save_name=f"{sample.sample_id}_n_neighbors_sweep.png",
)


### Sweep Leiden Resolution

After choosing a neighbor graph, resolution is the main granularity knob. Prefer a resolution where clusters are not mostly tiny fragments and where spatial domains or known biology remain interpretable.


In [ ]:
resolution_grid = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
resolution_results = []
for resolution in resolution_grid:
    resolution_results.append((f"resolution={resolution}", cluster_with(leiden_resolution=resolution)))

_, resolution_summary = plot_umap_and_spatial_grid(
    resolution_results,
    title="Leiden resolution sweep",
    save_name=f"{sample.sample_id}_leiden_resolution_sweep.png",
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(resolution_summary["leiden_resolution"], resolution_summary["n_clusters"], marker="o")
ax.set_xlabel("Leiden resolution")
ax.set_ylabel("Number of clusters")
ax.set_title("Resolution vs cluster count")
fig.tight_layout()
fig.savefig(tuning_dir / f"{sample.sample_id}_resolution_cluster_count.png", dpi=cfg.figure_dpi, bbox_inches="tight")
display(fig)


### Sweep UMAP `min_dist` and `spread`

This changes the 2D display, not the Leiden labels. Use it to make plots readable without treating distances between UMAP islands as quantitative biology.


In [ ]:
min_dist_grid = [0.05, 0.2, 0.5, 0.8]
spread_grid = [0.8, 1.0, 1.5]
umap_results = []
for min_dist, spread in product(min_dist_grid, spread_grid):
    if min_dist > spread:
        continue
    label = f"min_dist={min_dist}, spread={spread}"
    umap_results.append(
        (
            label,
            cluster_with(umap_min_dist=min_dist, umap_spread=spread),
        )
    )

fig, axes = plt.subplots(
    len(min_dist_grid),
    len(spread_grid),
    figsize=(4 * len(spread_grid), 3.8 * len(min_dist_grid)),
    squeeze=False,
)
result_by_label = {label: clustered_i for label, clustered_i in umap_results}
for i, min_dist in enumerate(min_dist_grid):
    for j, spread in enumerate(spread_grid):
        ax = axes[i, j]
        label = f"min_dist={min_dist}, spread={spread}"
        if label not in result_by_label:
            ax.axis("off")
            continue
        sc.pl.umap(
            result_by_label[label],
            color="leiden",
            ax=ax,
            title=label,
            legend_loc=None,
            show=False,
        )
fig.tight_layout()
fig.savefig(tuning_dir / f"{sample.sample_id}_umap_min_dist_spread_grid.png", dpi=cfg.figure_dpi, bbox_inches="tight")
display(fig)


### Normalize Target Sum and Highly Expressed Genes

For targeted Xenium/MERSCOPE panels, start with `exclude_highly_expressed=False`. Compare these only if QC suggests one or a few genes dominate size factors. `target_sum=None` uses the median count depth; `target_sum=1e4` gives CP10K-style scaling.


In [ ]:
normalization_grid = [
    ("median, include all", None, False),
    ("CP10K, include all", 1e4, False),
    ("median, exclude highly expressed", None, True),
    ("CP10K, exclude highly expressed", 1e4, True),
]
normalization_results = []
for label, target_sum, exclude_he in normalization_grid:
    normalization_results.append(
        (
            label,
            cluster_with(
                normalize_target_sum=target_sum,
                normalize_exclude_highly_expressed=exclude_he,
            ),
        )
    )

_, normalization_summary = plot_umap_and_spatial_grid(
    normalization_results,
    title="Normalization sweep",
    save_name=f"{sample.sample_id}_normalization_sweep.png",
)


### Stability Check Across Random Seeds

Use this after narrowing candidate settings. Higher mean pairwise adjusted Rand index means the Leiden assignments are less sensitive to the UMAP/PCA/Leiden random seed. This can take a few minutes because it reruns clustering several times per candidate.


In [ ]:
candidate_params = [
    {"label": "baseline", **base_params},
    {"label": "coarser", **(base_params | {"leiden_resolution": 0.3, "n_neighbors": 30})},
    {"label": "finer", **(base_params | {"leiden_resolution": 0.8, "n_neighbors": 15})},
]
stability_seeds = [0, 1, 2]
stability_rows = []
for candidate in candidate_params:
    labels_by_seed = []
    for seed in stability_seeds:
        params = {k: v for k, v in candidate.items() if k != "label"}
        params["random_seed"] = seed
        clustered_seed = run_scanpy_clustering(adata, **params)
        labels_by_seed.append(clustered_seed.obs["leiden"].astype(str).to_numpy())
    stability_rows.append(
        {
            "label": candidate["label"],
            "mean_pairwise_ari": mean_pairwise_ari(labels_by_seed),
            "seeds": str(stability_seeds),
            "n_pcs": candidate["n_pcs"],
            "n_neighbors": candidate["n_neighbors"],
            "leiden_resolution": candidate["leiden_resolution"],
            "umap_min_dist": candidate["umap_min_dist"],
            "umap_spread": candidate["umap_spread"],
        }
    )

pd.DataFrame(stability_rows).sort_values("mean_pairwise_ari", ascending=False)


Run the full pair stage function with the same config object.

In [ ]:
run_clustering_squidpy(cfg)